## Point Mass Lap Time Simulation — Monza 2023

### Context

Lap time simulation is a key tool in motorsport engineering. 
Being capable of testing different car settings virtually — 
without real track testing — provides a significant competitive 
advantage. It prevents excessive testing costs and saves 
valuable time during practice sessions by doing the setup 
work beforehand.

As Segers states (Ch.15.3): "Once the model validation is 
judged satisfactory, the model can be used to explore 
alternative vehicle configurations" — such as aerodynamic 
setup changes, weight reduction, or power unit upgrades.

This notebook implements a Point Mass Lap Time Simulation 
model applied to Carlos Sainz's fastest lap during the 
2023 Monza Grand Prix Qualifying (1:21.046). The model 
is built entirely from publicly available F1 2023 vehicle 
parameters and real telemetry data extracted via FastF1.

The simulation is structured in three steps :
1. **Mathematical model** — analytical derivation of the 
   governing equations using SymPy
2. **Track model** — calibration of effective corner radii 
   from real FastF1 telemetry
3. **Steady-state simulation** — lap time computation 
   segment by segment

In [136]:
import fastf1
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sympy import *

In [ ]:
session = fastf1.get_session(2023, 'Monza', 'Q')
session.load()
tel = session.laps.pick_drivers('SAI').pick_fastest().get_telemetry()


### Mathematical Model

We use a Point Mass Model for its simplicity — all vehicle 
mass is concentrated at a single point (center of gravity).
Despite this simplification, Segers (p.406) confirms it 
"can be used to gain basic knowledge of the car."

The following forces are applied to the point mass :

**Aerodynamic downforce :**
F_downforce = 0.5 × ρ × Cl × A × v²

**Aerodynamic drag :**
F_drag = 0.5 × ρ × Cx × A × v²

**Total normal force (grip) :**
F_normal = m×g + F_downforce

**Maximum cornering speed** (derived analytically with SymPy) :
v_max = √(μ×m×g / (m/r - 0.5×μ×ρ×Cl×A))

In [138]:
from sympy import *


v, r, mu, g, m, Cl, Cx, A, rho, P = symbols(
    'v r mu g m Cl Cx A rho P', positive=True)


F_weight    = m * g
F_downforce = Rational(1,2) * rho * Cl * A * v**2
F_drag      = Rational(1,2) * rho * Cx * A * v**2
F_normal    = F_weight + F_downforce
F_tire      = mu * F_normal

print("Normal Force :")
display(F_normal)

print("\nGrip :")
display(F_tire)

print("\nDrag Force :")
display(F_drag)

Normal Force :


A*Cl*rho*v**2/2 + g*m


Grip :


mu*(A*Cl*rho*v**2/2 + g*m)


Drag Force :


A*Cx*rho*v**2/2

In [139]:

# Centrifugal Force = Lateral Grip
centripetal  = m * v**2 / r
lateral_grip = mu * F_normal

equation = Eq(centripetal, lateral_grip)

print("Equation of equilibrium :")
display(equation)


# m*v²/r = mu*(m*g + 0.5*rho*Cl*A*v²)
# v²*(m/r - 0.5*mu*rho*Cl*A) = mu*m*g
v_max_expr = sqrt(mu * m * g / 
                  (m/r - Rational(1,2) * mu * rho * Cl * A))

print("\nMaximum speed in corner :")
display(v_max_expr)

# Numerical verification — Parabolica r=250m
v_max_num = v_max_expr.subs({
    mu: 1.8, m: 798, g: 9.81,
    r: 250, rho: 1.225, Cl: 3.7, A: 1.5
})

Equation of equilibrium :


Eq(m*v**2/r, mu*(A*Cl*rho*v**2/2 + g*m))


Maximum speed in corner :


sqrt(g)*sqrt(m)*sqrt(mu)*sqrt(1/(-A*Cl*mu*rho/2 + m/r))

In [140]:
# Maximum engine force (limitate by engine power or grip)
F_engine_power = P / v          # limitate by engine power
F_engine_grip  = mu * F_normal  # limitate by tire grip

print("Maximum engine force (limitate by engine power) :")
display(F_engine_power)

print("\nEngine force (limitate by tire grip) :")
display(F_engine_grip)

# Acceleration 
F_rolling = Rational(1,100) * m * g  # Rx = 0.01
a_net = (F_engine_power - F_drag - F_rolling) / m

print("\nAcceleracion :")
display(simplify(a_net))

# Maximum deceleration at braking
a_brake = -mu * (g + F_downforce/m)
print("\nMaximum deceleration at braking :")
display(simplify(a_brake))

Maximum engine force (limitate by engine power) :


P/v


Engine force (limitate by tire grip) :


mu*(A*Cl*rho*v**2/2 + g*m)


Acceleracion :


-A*Cx*rho*v**2/(2*m) + P/(m*v) - g/100


Maximum deceleration at braking :


-A*Cl*mu*rho*v**2/(2*m) - g*mu

### Track Model & Calibration Strategy

The circuit is divided into 6 corner segments identified 
from the braking analysis and 2 straight segments.

**Effective corner radius calibration :**
Rather than using GPS-derived radii (too noisy), we 
calibrate an effective radius r_eff for each corner using 
the real minimum corner speed from FastF1 telemetry and 
reference vehicle parameters. This radius is then used 
to recalculate v_min dynamically when parameters change.

| Corner | v_min real | r_eff |
|--------|-----------|-------|
| Variante del Rettifilo | 70 km/h | 19.3m |
| Lesmo 1 | 112 km/h | 43.8m |
| Lesmo 2 | 179 km/h | 91.7m |
| Variante Ascari | 179 km/h | 91.7m |
| Variante Ascari 2 | 196 km/h | 104.7m |
| Parabolica | 206 km/h | 112.5m |

In [141]:
for bp, d_min, v_min in corner_speeds:
    idx_vmin = (tel['Distance'] - d_min).abs().idxmin()
    tel_after = tel[tel['Distance'] > d_min]
    idx_throttle = tel_after[tel_after['Throttle'] >= 95].index[0]
    d_throttle = tel['Distance'].loc[idx_throttle]
    corner_len = d_throttle - d_min
    print(f"Distance {d_min:.0f}m → turn exit {d_throttle:.0f}m → reel length = {corner_len:.0f}m")

Distance 941m → turn exit 1005m → reel length = 64m
Distance 2122m → turn exit 2192m → reel length = 70m
Distance 2849m → turn exit 2876m → reel length = 27m
Distance 2849m → turn exit 2876m → reel length = 27m
Distance 3915m → turn exit 3957m → reel length = 42m
Distance 5149m → turn exit 5186m → reel length = 38m


### Simulation Methodology

Each corner segment is simulated in 3 phases 
with a time step dt = 0.01s :

**Phase 1 — Braking :**
Deceleration combines tyre friction and aerodynamic drag :
a = -(μ × F_normal / m) - (F_drag / m)
→ Higher speed = more downforce = harder braking possible

**Phase 2 — Cornering :**
Constant speed = v_min over the corner length
t_corner = corner_length / v_min
Corner length extracted from FastF1 : distance between 
v_min point and throttle ≥ 95% point.

**Phase 3 — Acceleration :**
Engine force limited by power or tyre grip :
F_engine = min(P_max/v, μ × F_normal)
a = (F_engine - F_drag - F_rolling) / m

In [142]:
# ═══════════════════════════════════════════════════
# VEHICLE PARAMETERS F1 2023 — MODIFY HERE
# ═══════════════════════════════════════════════════

mass_n  = 798       # kg  — car + driver mass
P_max_n = 735000    # W   — max power (1000 HP)
Cx_n    = 0.85      # —   — drag coefficient
Cl_n    = 3.7       # —   — downforce coefficient
A_n     = 1.5       # m²  — frontal area
rho_n   = 1.225     # kg/m³ — air density
mu_n    = 1.8       # —   — tyre friction coefficient
Rx_n    = 0.005     # —   — rolling resistance
g_n     = 9.81      # m/s²
dt      = 0.01      # s   — simulation time step

print("✅ Parameters loaded :")
print(f"   Mass     : {mass_n} kg")
print(f"   Power    : {P_max_n/1000:.0f} kW ({P_max_n/745.7:.0f} HP)")
print(f"   Cx       : {Cx_n}")
print(f"   Cl       : {Cl_n}")
print(f"   μ        : {mu_n}")

# ═══════════════════════════════════════════════════
# REFERENCE PARAMETERS — DO NOT MODIFY
# (used to calibrate r_eff from real FastF1 data)
# ═══════════════════════════════════════════════════

mu_ref   = 1.8
Cl_ref   = 3.7
rho_ref  = 1.225
A_ref    = 1.5
g_ref    = 9.81
mass_ref = 798

# ═══════════════════════════════════════════════════
# CIRCUIT SEGMENTS — extracted from WE23 analysis
# ═══════════════════════════════════════════════════

circuit_segments = [
    {'name': 'Variante del Rettifilo', 'brake_dist': 782,  'v_min_real': 70.0,  'corner_length': 64},
    {'name': 'Lesmo 1',                'brake_dist': 2011, 'v_min_real': 112.0, 'corner_length': 70},
    {'name': 'Lesmo 2',                'brake_dist': 2434, 'v_min_real': 179.0, 'corner_length': 27},
    {'name': 'Variante Ascari',        'brake_dist': 2771, 'v_min_real': 179.0, 'corner_length': 27},
    {'name': 'Variante Ascari 2',      'brake_dist': 3838, 'v_min_real': 196.0, 'corner_length': 42},
    {'name': 'Parabolica',             'brake_dist': 5049, 'v_min_real': 206.0, 'corner_length': 38},
]

total_length    = 5793  # m — Monza circuit length
d_missing_start = circuit_segments[0]['brake_dist']
d_missing_end   = total_length - (circuit_segments[-1]['brake_dist'] +
                                   circuit_segments[-1]['corner_length'])

# ═══════════════════════════════════════════════════
# STEP 1 — Calibrate effective corner radius r_eff
# using REFERENCE parameters and real FastF1 v_min
# ═══════════════════════════════════════════════════

print("\nEffective corner radius calibration (reference parameters):\n")
for seg in circuit_segments:
    v_ms    = seg['v_min_real'] / 3.6

    # Solve quadratic equation for r_eff :
    # v² = mu*g*r + mu*g*0.5*rho*Cl*A/m * r²
    # → (mu*g*0.5*rho*Cl*A/m)*r² + (mu*g)*r - v² = 0
    a_coeff = mu_ref * g_ref * 0.5 * rho_ref * Cl_ref * A_ref / mass_ref
    b_coeff = mu_ref * g_ref
    c_coeff = -v_ms**2

    r_eff        = (-b_coeff + np.sqrt(b_coeff**2 - 4*a_coeff*c_coeff)) / (2*a_coeff)
    seg['r_eff'] = r_eff
    print(f"{seg['name']:<25} → r_eff = {r_eff:.1f}m")

# ═══════════════════════════════════════════════════
# STEP 2 — Recalculate v_min with NEW parameters
# → allows Cx, Cl, mu variation
# ═══════════════════════════════════════════════════

print("\nv_min recalculation with new parameters:\n")
for seg in circuit_segments:
    r            = seg['r_eff']
    v_min_calc   = np.sqrt(mu_n * g_n * r *
                           (1 + 0.5*rho_n*Cl_n*A_n*r/mass_n))
    seg['v_min'] = v_min_calc * 3.6
    print(f"{seg['name']:<25} → v_min_real={seg['v_min_real']:.1f} → v_min_new={seg['v_min']:.1f} km/h")

# ═══════════════════════════════════════════════════
# STEP 3 — FULL LAP SIMULATION
# ═══════════════════════════════════════════════════

total_time    = 0
segment_times = []

# ── Segment 0 : Main straight (0 → first brake point) ───
v        = circuit_segments[-1]['v_min'] / 3.6
t_accel  = 0
d_accel  = 0
d_target = d_missing_start

while d_accel < d_target:
    F_drag   = 0.5 * rho_n * Cx_n * A_n * v**2
    F_roll   = Rx_n * mass_n * g_n
    F_engine = min(P_max_n / v, mu_n * mass_n * g_n)
    a        = (F_engine - F_drag - F_roll) / mass_n
    v       += a * dt
    d_accel += v * dt
    t_accel += dt

total_time += t_accel
v_start     = v
print(f"\nMain straight : accel={t_accel:.2f}s → v={v*3.6:.1f} km/h")

# ── Corner segments ──────────────────────────────────────
for i, seg in enumerate(circuit_segments):

    v_min = seg['v_min'] / 3.6

    # Phase 1 — BRAKING
    # Deceleration = tyre friction + aero drag assistance
    v       = v_start
    t_brake = 0

    while v > v_min:
        F_drag      = 0.5 * rho_n * Cx_n * A_n * v**2
        F_downforce = 0.5 * rho_n * Cl_n * A_n * v**2
        F_normal    = mass_n * g_n + F_downforce
        a           = -(mu_n * F_normal / mass_n) - (F_drag / mass_n)
        v          += a * dt
        v           = max(v, v_min)
        t_brake    += dt

    # Phase 2 — CORNERING
    # Constant speed = v_min over corner length
    t_corner = seg['corner_length'] / v_min

    # Phase 3 — ACCELERATION
    # Engine force limited by power or tyre grip
    v       = v_min
    t_accel = 0
    d_accel = 0

    if i < len(circuit_segments) - 1:
        d_target = (circuit_segments[i+1]['brake_dist'] -
                    seg['brake_dist'] -
                    seg['corner_length'])
    else:
        d_target = 0

    while d_accel < d_target:
        F_drag   = 0.5 * rho_n * Cx_n * A_n * v**2
        F_roll   = Rx_n * mass_n * g_n
        F_engine = min(P_max_n / v, mu_n * mass_n * g_n)
        a        = (F_engine - F_drag - F_roll) / mass_n
        v       += a * dt
        d_accel += v * dt
        t_accel += dt

    v_start    = v
    t_segment  = t_brake + t_corner + t_accel
    total_time += t_segment

    segment_times.append({
        'name'    : seg['name'],
        't_brake' : t_brake,
        't_corner': t_corner,
        't_accel' : t_accel,
        't_total' : t_segment
    })

    print(f"{seg['name']:<25} : brake={t_brake:.2f}s | corner={t_corner:.2f}s | accel={t_accel:.2f}s | total={t_segment:.2f}s")

# ── Final segment : Parabolica exit ─────────────────────
v        = circuit_segments[-1]['v_min'] / 3.6
t_accel  = 0
d_accel  = 0
d_target = d_missing_end

while d_accel < d_target:
    F_drag   = 0.5 * rho_n * Cx_n * A_n * v**2
    F_roll   = Rx_n * mass_n * g_n
    F_engine = min(P_max_n / v, mu_n * mass_n * g_n)
    a        = (F_engine - F_drag - F_roll) / mass_n
    v       += a * dt
    d_accel += v * dt
    t_accel += dt

total_time += t_accel
print(f"Parabolica exit     : accel={t_accel:.2f}s → v={v*3.6:.1f} km/h")

# ═══════════════════════════════════════════════════
# FINAL RESULTS
# ═══════════════════════════════════════════════════

print(f"\n{'═'*55}")
print(f"🏁 Simulated lap time : {total_time:.3f}s = {int(total_time//60)}:{total_time%60:.3f}")
print(f"🏁 Real lap time SAI  : 1:21.046")
print(f"   Gap               : {total_time - 81.046:+.3f}s ({abs((total_time-81.046)/81.046)*100:.2f}%)")
print(f"{'═'*55}")

✅ Parameters loaded :
   Mass     : 798 kg
   Power    : 735 kW (986 HP)
   Cx       : 0.85
   Cl       : 3.7
   μ        : 1.8

Effective corner radius calibration (reference parameters):

Variante del Rettifilo    → r_eff = 19.8m
Lesmo 1                   → r_eff = 45.9m
Lesmo 2                   → r_eff = 98.6m
Variante Ascari           → r_eff = 98.6m
Variante Ascari 2         → r_eff = 113.2m
Parabolica                → r_eff = 122.0m

v_min recalculation with new parameters:

Variante del Rettifilo    → v_min_real=70.0 → v_min_new=70.0 km/h
Lesmo 1                   → v_min_real=112.0 → v_min_new=112.0 km/h
Lesmo 2                   → v_min_real=179.0 → v_min_new=179.0 km/h
Variante Ascari           → v_min_real=179.0 → v_min_new=179.0 km/h
Variante Ascari 2         → v_min_real=196.0 → v_min_new=196.0 km/h
Parabolica                → v_min_real=206.0 → v_min_new=206.0 km/h

Main straight : accel=9.30s → v=342.5 km/h
Variante del Rettifilo    : brake=1.84s | corner=3.29s | accel=

### Results

| Segment | Braking | Cornering | Accel | Total |
|---------|---------|-----------|-------|-------|
| Main straight | - | - | 9.30s | 9.30s |
| Variante del Rettifilo | 1.95s | 3.29s | 14.70s | 19.94s |
| Lesmo 1 | 1.45s | 2.25s | 5.46s | 9.16s |
| Lesmo 2 | 0.69s | 0.54s | 4.35s | 5.58s |
| Variante Ascari | 0.69s | 0.54s | 12.23s | 13.46s |
| Variante Ascari 2 | 0.71s | 0.77s | 13.41s | 14.89s |
| Parabolica | 0.65s | 0.66s | 0.00s | 1.31s |
| Parabolica exit | - | - | 8.50s | 8.50s |
| **TOTAL** | | | | **81.733s** |

🏁 Simulated lap time : 1:21.733  
🏁 Real lap time SAI  : 1:21.046  
Gap : +0.687s (0.85% error)

### Limitations

- **Estimated parameters** : Cx, Cl, P_max and μ are 
  public estimates — real values are confidential. 
  Calibration will be performed .

- **Constant μ** : in reality μ varies with tyre 
  temperature, speed and degradation (Pacejka model).

- **No DRS** : DRS reduces Cx by ~15% on straights — 
  not implemented in this version.

- **GPS track model** : GPS-derived radii were too noisy 
  for direct use — replaced by r_eff calibrated from 
  real FastF1 corner speeds.

- **Fixed corner geometry** : brake points and corner 
  lengths are extracted from SAI's real data — the model 
  cannot yet optimise trajectory automatically.

- **No track elevation** : Monza is relatively flat but 
  elevation changes affect acceleration and braking 
  (Segers p.81).